In [0]:
select * from datamodeling.silver.silver_table

##Dim Customers

In [0]:
--creating a dimension table (don't apply on duplicate values)
create or replace table datamodeling.gold.dimCustomer
as(
select distinct(customer_id), 
row_number() over(order by customer_id) as dimCustomerKey, 
customer_name, 
customer_email, 
customer_name_upper from datamodeling.silver.silver_table)

In [0]:
select * from datamodeling.gold.dimcustomer

##Dim Products

In [0]:
--creating Product Dimension
create or replace table datamodeling.gold.dimProduct as (
select row_number() over (order by product_id) as dimProductKey, 
product_id, product_name, product_category 
from datamodeling.silver.silver_table)

In [0]:
select * from datamodeling.gold.dimproduct

## Dim Payments

In [0]:
create or replace table datamodeling.gold.dimPayment as(
with rm_dup as (
  select distinct(payment_type) from 
  datamodeling.silver.silver_table
)
select row_number() over (order by payment_type) as dimPaymentKey, payment_type from rm_dup)

In [0]:
select * from datamodeling.gold.dimpayment

## Dim Region

In [0]:
create or replace table datamodeling.gold.dimRegion
as (
  with rm_dup as(
    select distinct(country) from datamodeling.silver.silver_table
  )
  select row_number() over (order by country) as dimRegionKey, country 
  from rm_dup
)

In [0]:
select * from datamodeling.gold.dimRegion

##Dim Sales

In [0]:
%python
spark.sql("""select * from datamodeling.silver.silver_table""").columns

In [0]:
create or replace table datamodeling.gold.dimSales 
as(
select 
row_number() over (order by order_id) as dimSalesKey,
 order_id,
 order_date,
 customer_id,
 customer_name,
 customer_email,
 product_id,
 product_name,
 product_category,
 payment_type,
 country,
 last_update,
 customer_name_upper,
 processDate 
 from datamodeling.silver.silver_table)

In [0]:
select * from datamodeling.gold.dimSales

## FACT TABLE

In [0]:
create or replace table datamodeling.gold.factSales(
SELECT 
  S.dimSalesKey,
  C.dimCustomerKey,
  P.dimProductKey,
  R.dimRegionKey,
  Pa.dimPaymentKey,
  F.quantity,
  F.unit_price
FROM datamodeling.silver.silver_table F
LEFT JOIN datamodeling.gold.dimcustomer C
  ON F.customer_id = C.customer_id
LEFT JOIN datamodeling.gold.dimProduct P
  ON F.product_id = P.product_id
LEFT JOIN datamodeling.gold.dimRegion R
  ON F.country = R.country
LEFT JOIN datamodeling.gold.dimPayment Pa
  ON F.payment_type = Pa.payment_type
LEFT JOIN datamodeling.gold.dimSales S
  ON F.order_id = S.order_id)

In [0]:
select * from datamodeling.gold.factSales